# 08_reranking_experiment.ipynb

**Actividad de Grado MIA UC**  
**Sistema Inteligente de Vigilancia de Riesgos Documentales basado en RAG y LLMs**

Este notebook implementa y evalúa un experimento de **reranking** sobre los resultados del retriever BGE-M3.

## Objetivo
Comparar dos escenarios:

1. **Retriever base:** BGE-M3 + similitud coseno
2. **Retriever + Reranker:** BGE-M3 + CrossEncoder

## Arquitectura

```text
Pregunta
   ↓
Retriever BGE-M3 Top-N
   ↓
Reranker CrossEncoder
   ↓
Top-K reordenado
   ↓
Evaluación
```

## Entradas
- `chunks_recursive.parquet`
- `gold_questions.csv`

## Salidas
- `reranking_results.csv`
- `reranking_results.xlsx`
- `reranking_manual_evaluation_template.xlsx`
- `resumen_reranking_experiment.csv`


In [ ]:
# =====================================================
# 08_reranking_experiment.ipynb
# Actividad de Grado MIA UC
# Patricia Patiño
# =====================================================

!pip -q install pandas pyarrow sentence-transformers scikit-learn openpyxl

## 1. Importar librerías

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from google.colab import files
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_colwidth', 160)

print('Librerías cargadas correctamente')

## 2. Cargar archivos

Sube:
- `chunks_recursive.parquet`
- `gold_questions.csv`

In [ ]:
print('Sube chunks_recursive.parquet y gold_questions.csv')
uploaded = files.upload()
print('\nArchivos cargados:', list(uploaded.keys()))

## 3. Leer y validar datos

In [ ]:
chunks_path = Path('chunks_recursive.parquet')
questions_path = Path('gold_questions.csv')

if not chunks_path.exists():
    raise FileNotFoundError('No se encontró chunks_recursive.parquet')

if not questions_path.exists():
    raise FileNotFoundError('No se encontró gold_questions.csv')

chunks_df = pd.read_parquet(chunks_path).reset_index(drop=True)
gold_questions = pd.read_csv(questions_path).reset_index(drop=True)

required_chunks = {'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_text'}
required_questions = {'question_id', 'question', 'categoria'}

missing_chunks = required_chunks - set(chunks_df.columns)
missing_questions = required_questions - set(gold_questions.columns)

if missing_chunks:
    raise ValueError(f'Faltan columnas en chunks: {missing_chunks}')

if missing_questions:
    raise ValueError(f'Faltan columnas en gold_questions: {missing_questions}')

print('Chunks cargados:', len(chunks_df))
print('Preguntas cargadas:', len(gold_questions))
display(chunks_df.head(3))
display(gold_questions.head(3))

## 4. Configuración del experimento

- `TOP_N_RETRIEVER`: cantidad inicial de chunks recuperados por BGE-M3.
- `TOP_K_FINAL`: cantidad final después del reranking.

El reranker solo reordena los documentos candidatos; no busca en todo el corpus directamente.

In [ ]:
EMBEDDING_MODEL_NAME = 'BAAI/bge-m3'

# Reranker liviano y fácil de ejecutar en Colab.
# Aunque fue entrenado principalmente en inglés, sirve como primer baseline de reranking.
RERANKER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_N_RETRIEVER = 20
TOP_K_FINAL = 5

print('Embedding model:', EMBEDDING_MODEL_NAME)
print('Reranker model:', RERANKER_MODEL_NAME)
print('Top-N inicial:', TOP_N_RETRIEVER)
print('Top-K final:', TOP_K_FINAL)

## 5. Cargar modelos

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
reranker_model = CrossEncoder(RERANKER_MODEL_NAME)

print('Modelos cargados correctamente')

## 6. Generar embeddings del corpus

In [ ]:
chunk_texts = chunks_df['chunk_text'].astype(str).tolist()

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print('Embeddings generados:', chunk_embeddings.shape)

## 7. Funciones de retrieval y reranking

In [ ]:
def retrieve_candidates(question, top_n=20):
    """Recupera candidatos iniciales usando BGE-M3 + similitud coseno."""
    q_emb = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(q_emb, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_n]

    candidates = []
    for rank, idx in enumerate(top_indices, start=1):
        row = chunks_df.iloc[idx]
        candidates.append({
            'retriever_rank': rank,
            'retriever_score': float(scores[idx]),
            'chunk_index': int(idx),
            'doc_id': row['doc_id'],
            'filename': row['filename'],
            'tipo_documento': row['tipo_documento'],
            'page': row['page'],
            'chunk_id': row['chunk_id'],
            'chunk_text': row['chunk_text']
        })

    return pd.DataFrame(candidates)


def rerank_candidates(question, candidates_df, top_k=5):
    """Reordena candidatos usando CrossEncoder sobre pares pregunta-chunk."""
    pairs = [
        [question, str(text)]
        for text in candidates_df['chunk_text'].tolist()
    ]

    reranker_scores = reranker_model.predict(pairs)

    reranked = candidates_df.copy()
    reranked['reranker_score'] = reranker_scores
    reranked = reranked.sort_values('reranker_score', ascending=False).reset_index(drop=True)
    reranked['reranker_rank'] = range(1, len(reranked) + 1)

    return reranked.head(top_k).copy()


def run_reranking_experiment(question_id, question, categoria):
    candidates = retrieve_candidates(question, top_n=TOP_N_RETRIEVER)

    # Escenario A: retriever base, top-k directo
    base_topk = candidates.head(TOP_K_FINAL).copy()
    base_topk['scenario'] = 'retriever_only'
    base_topk['final_rank'] = base_topk['retriever_rank']
    base_topk['final_score'] = base_topk['retriever_score']
    base_topk['reranker_rank'] = np.nan
    base_topk['reranker_score'] = np.nan

    # Escenario B: retriever + reranker
    reranked_topk = rerank_candidates(question, candidates, top_k=TOP_K_FINAL)
    reranked_topk['scenario'] = 'retriever_plus_reranker'
    reranked_topk['final_rank'] = reranked_topk['reranker_rank']
    reranked_topk['final_score'] = reranked_topk['reranker_score']

    combined = pd.concat([base_topk, reranked_topk], ignore_index=True)
    combined['question_id'] = question_id
    combined['question'] = question
    combined['categoria'] = categoria

    cols = [
        'scenario', 'question_id', 'question', 'categoria',
        'final_rank', 'final_score',
        'retriever_rank', 'retriever_score',
        'reranker_rank', 'reranker_score',
        'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id', 'chunk_text'
    ]

    return combined[cols]

## 8. Ejecutar experimento sobre preguntas prioritarias

In [ ]:
# Puedes ampliar esta lista después a las 20 preguntas.
preguntas_prioritarias = [
    'Q001', 'Q002', 'Q003', 'Q004', 'Q005',
    'Q006', 'Q010', 'Q012', 'Q013', 'Q020'
]

questions_to_run = gold_questions[gold_questions['question_id'].isin(preguntas_prioritarias)].copy()

all_results = []

for _, row in questions_to_run.iterrows():
    print('Ejecutando:', row['question_id'], row['question'])
    result = run_reranking_experiment(
        question_id=row['question_id'],
        question=row['question'],
        categoria=row['categoria']
    )
    all_results.append(result)

reranking_results = pd.concat(all_results, ignore_index=True)

print('Resultados generados:', len(reranking_results))
display(reranking_results.head(10))

## 9. Comparación visual para una pregunta

Esta celda permite revisar cómo cambia el orden entre el retriever base y el reranker.

In [ ]:
QUESTION_ID_TO_REVIEW = 'Q001'

for scenario in ['retriever_only', 'retriever_plus_reranker']:
    print('='*100)
    print('Escenario:', scenario)
    print('='*100)

    subset = reranking_results[
        (reranking_results['question_id'] == QUESTION_ID_TO_REVIEW) &
        (reranking_results['scenario'] == scenario)
    ].sort_values('final_rank')

    for _, row in subset.iterrows():
        print(f"Rank {int(row['final_rank'])} | Retriever rank {int(row['retriever_rank'])} | {row['filename']} | {row['chunk_id']}")
        print('Retriever score:', round(row['retriever_score'], 4), '| Reranker score:', row['reranker_score'])
        print(row['chunk_text'][:700])
        print('-'*100)

## 10. Crear plantilla de evaluación manual

Marca `relevante_manual` con:
- `1`: el chunk ayuda a responder la pregunta.
- `0`: el chunk no ayuda a responder la pregunta.

Esto permite comparar Precision@K, Hit@K y MRR entre los dos escenarios.

In [ ]:
reranking_eval_template = reranking_results.copy()
reranking_eval_template['relevante_manual'] = ''
reranking_eval_template['comentario_evaluador'] = ''

eval_cols = [
    'scenario', 'question_id', 'question', 'categoria',
    'final_rank', 'final_score',
    'retriever_rank', 'retriever_score', 'reranker_rank', 'reranker_score',
    'doc_id', 'filename', 'tipo_documento', 'page', 'chunk_id',
    'relevante_manual', 'comentario_evaluador', 'chunk_text'
]

reranking_eval_template = reranking_eval_template[eval_cols]

display(reranking_eval_template.head())

## 11. Evaluación automática opcional si ya tienes relevancia manual

Si después completas el Excel con 0/1, puedes volver a subirlo en esta sección para calcular métricas.

In [ ]:
def precision_at_k(group, k):
    topk = group[group['final_rank'] <= k]
    if len(topk) == 0:
        return 0
    return topk['relevante_manual'].mean()


def hit_at_k(group, k):
    topk = group[group['final_rank'] <= k]
    if len(topk) == 0:
        return 0
    return int(topk['relevante_manual'].sum() > 0)


def mrr(group):
    ordered = group.sort_values('final_rank')
    relevant = ordered[ordered['relevante_manual'] == 1]
    if relevant.empty:
        return 0
    first_rank = relevant['final_rank'].min()
    return 1 / first_rank


def calcular_metricas(evaluated_df):
    rows = []
    for (scenario, question_id), group in evaluated_df.groupby(['scenario', 'question_id']):
        rows.append({
            'scenario': scenario,
            'question_id': question_id,
            'question': group['question'].iloc[0],
            'categoria': group['categoria'].iloc[0],
            'precision_at_1': precision_at_k(group, 1),
            'precision_at_3': precision_at_k(group, 3),
            'precision_at_5': precision_at_k(group, 5),
            'hit_at_1': hit_at_k(group, 1),
            'hit_at_3': hit_at_k(group, 3),
            'hit_at_5': hit_at_k(group, 5),
            'mrr': mrr(group)
        })

    by_question = pd.DataFrame(rows)
    by_scenario = (
        by_question.groupby('scenario')
        .agg(
            questions_evaluated=('question_id', 'nunique'),
            precision_at_1=('precision_at_1', 'mean'),
            precision_at_3=('precision_at_3', 'mean'),
            precision_at_5=('precision_at_5', 'mean'),
            hit_at_1=('hit_at_1', 'mean'),
            hit_at_3=('hit_at_3', 'mean'),
            hit_at_5=('hit_at_5', 'mean'),
            mrr=('mrr', 'mean')
        )
        .reset_index()
    )
    return by_question, by_scenario

## 12. Resumen técnico del experimento

In [ ]:
resumen_reranking = pd.DataFrame([
    {
        'embedding_model': EMBEDDING_MODEL_NAME,
        'reranker_model': RERANKER_MODEL_NAME,
        'num_chunks': len(chunks_df),
        'num_questions': len(questions_to_run),
        'top_n_retriever': TOP_N_RETRIEVER,
        'top_k_final': TOP_K_FINAL,
        'scenarios': 'retriever_only; retriever_plus_reranker'
    }
])

display(resumen_reranking)

## 13. Guardar artefactos

In [ ]:
reranking_results.to_csv('reranking_results.csv', index=False, encoding='utf-8-sig')
reranking_results.to_excel('reranking_results.xlsx', index=False)
reranking_eval_template.to_excel('reranking_manual_evaluation_template.xlsx', index=False)
resumen_reranking.to_csv('resumen_reranking_experiment.csv', index=False, encoding='utf-8-sig')

print('Archivos generados:')
print('- reranking_results.csv')
print('- reranking_results.xlsx')
print('- reranking_manual_evaluation_template.xlsx')
print('- resumen_reranking_experiment.csv')

## 14. Descargar artefactos

In [ ]:
files.download('reranking_results.csv')
files.download('reranking_results.xlsx')
files.download('reranking_manual_evaluation_template.xlsx')
files.download('resumen_reranking_experiment.csv')